# BICEP Basic Analysis Tutorial

This notebook demonstrates how to run BICEP analyses to estimate electrical infrastructure upgrade costs for different geographic scopes.

**Prerequisites**: 
- BICEP must be installed: From the BICEP root directory, run `pip install -e .`
- The notebook kernel should use the `bicep-env` conda environment

## Setup

First, import the BICEP analysis module:

In [1]:
# Configure loguru FIRST to suppress verbose logging in notebooks only
import sys
import os
import warnings
from loguru import logger

# Suppress specific pandas FutureWarning about downcasting
warnings.filterwarnings('ignore', message='.*Downcasting object dtype arrays.*')

logger.remove()  # Remove default handler

# Add filter to suppress tech_adoption warnings
def filter_tech_warnings(record):
    return not any(msg in record["message"] for msg in [
        "No data found for technology",
        "No growth data available"
    ])

logger.add(sys.stderr, level='WARNING', format='<level>{level: <8}</level> | <cyan>{name}</cyan>:<cyan>{function}</cyan>:<cyan>{line}</cyan> - <level>{message}</level>', filter=filter_tech_warnings)

from pathlib import Path

# Add BICEP to Python path
# Get the BICEP root directory (three levels up from this notebook)
bicep_root = Path.cwd().parent.parent.parent
if str(bicep_root) not in sys.path:
    sys.path.insert(0, str(bicep_root))

print(f"BICEP root: {bicep_root}")
print("Python path updated. You can now import bicep modules.")

BICEP root: /Users/faye994/code/BICEP
Python path updated. You can now import bicep modules.


In [2]:
import os

# SUPPRESS LOGGING FOR CLEAN NOTEBOOK OUTPUT
# To see full BICEP logging details, comment out the line below:
os.environ['LOGURU_LEVEL'] = 'WARNING'

In [3]:
from bicep.analysis import BicepResults, BicepMultiStateResults

## Example 1: All States Analysis

Load pre-computed results for the Business-As-Usual (BAU) scenario across all US states using local SQLite database:

In [4]:
# Run analysis for BAU scenario across all US states
# This uses individual state analysis to avoid national competition effects
print("Running BAU scenario analysis for all US states...")
print("(This may take a few minutes as it processes each state separately)\n")

bau_all = BicepMultiStateResults(scenario='bau', mode='local', target_states='all', save_results=True)

Running BAU scenario analysis for all US states...
(This may take a few minutes as it processes each state separately)



### View Total Cost

Get the total infrastructure upgrade cost across all states and years:

### Visualize Cost Drivers

Plot the breakdown of costs by different upgrade drivers (EVs, heat pumps, solar, load growth):

In [5]:
print("="*70)
print("ALL STATES - BAU SCENARIO RESULTS")
print("="*70)
print(f'\nTotal BAU cost (all states): ${bau_all.total_cost:,.0f}')
print(f'Residential costs: ${bau_all.total_residential_costs:,.0f}')
print(f'Commercial costs: ${bau_all.total_commercial_costs:,.0f}')

print(f'\nTotal buildings analyzed: {len(bau_all.all_states_buildings):,}')
print(f'States included: {len(bau_all.all_states_buildings["state"].unique())} states')

ALL STATES - BAU SCENARIO RESULTS

Total BAU cost (all states): $5,871,515,140
Residential costs: $5,271,184,954
Commercial costs: $600,330,186

Total buildings analyzed: 884,596
States included: 51 states


In [6]:
# Show cost breakdown by state
print("\nTop 10 States by Total Cost:")
state_costs = bau_all.all_states_results.sort_values('total_cost', ascending=False)
for idx, row in state_costs.head(10).iterrows():
    print(f"  {row['state']}: ${row['total_cost']:,.0f}")


Top 10 States by Total Cost:
  CA: $1,008,585,119
  TX: $408,206,744
  MI: $286,475,080
  FL: $269,658,262
  NY: $212,095,937
  NJ: $204,291,592
  NC: $200,583,883
  IL: $172,787,296
  CO: $170,968,514
  WA: $168,481,591


## Example 2: Single State Analysis

Analyze a single state (California) to see state-specific infrastructure costs:

In [7]:
# Analyze just California
print("Running BAU scenario analysis for California...")
bau_ca = BicepResults(scenario='bau', mode='local', target_states='CA')

Running BAU scenario analysis for California...


### California Results

In [8]:
print("="*70)
print("CALIFORNIA - BAU SCENARIO RESULTS")
print("="*70)
print(f'\nTotal BAU cost (California only): ${bau_ca.total_cost:,.0f}')
print(f'Residential costs: ${bau_ca.total_residential_costs:,.0f}')
print(f'Commercial costs: ${bau_ca.total_commercial_costs:,.0f}')

print(f'\nNumber of buildings analyzed: {len(bau_ca.buildings):,}')
print(f'Buildings requiring upgrades: {len(bau_ca.buildings[bau_ca.buildings["upgrade_required"] == 1]):,}')

CALIFORNIA - BAU SCENARIO RESULTS

Total BAU cost (California only): $999,904,710
Residential costs: $911,334,719
Commercial costs: $88,569,991

Number of buildings analyzed: 96,923
Buildings requiring upgrades: 18,009


In [9]:
# Show cost statistics
print("\nCost Distribution Analysis (California):")
print(f"  Mean: ${bau_ca.buildings['weighted_cost'].mean():,.0f}")
print(f"  Median: ${bau_ca.buildings['weighted_cost'].median():,.0f}")
print(f"  Min: ${bau_ca.buildings['weighted_cost'].min():,.0f}")
print(f"  Max: ${bau_ca.buildings['weighted_cost'].max():,.0f}")


Cost Distribution Analysis (California):
  Mean: $55,523
  Median: $27,278
  Min: $313
  Max: $498,265


## Example 3: Multiple States Analysis

Analyze a specific set of states (California, Texas, and Washington) together:

In [10]:
# Analyze California, Texas, and Washington together
print("Running BAU scenario analysis for CA, TX, WA...")
bau_multi = BicepMultiStateResults(scenario='bau', mode='local', target_states=['CA', 'TX', 'WA'], save_results=False)

Running BAU scenario analysis for CA, TX, WA...


### Multi-State Results

In [11]:
print("="*70)
print("MULTI-STATE RESULTS (CA, TX, WA) - BAU SCENARIO")
print("="*70)
print(f'\nTotal BAU cost (CA, TX, WA): ${bau_multi.total_cost:,.0f}')
print(f'Residential costs: ${bau_multi.total_residential_costs:,.0f}')
print(f'Commercial costs: ${bau_multi.total_commercial_costs:,.0f}')

# Break down costs by state
print('\nCosts by state:')
state_costs = bau_multi.all_states_buildings.groupby('state')['weighted_cost'].sum()
for state in sorted(state_costs.index):
    print(f'  {state}: ${state_costs[state]:,.0f}')

MULTI-STATE RESULTS (CA, TX, WA) - BAU SCENARIO

Total BAU cost (CA, TX, WA): $1,541,046,559
Residential costs: $1,386,759,109
Commercial costs: $154,287,450

Costs by state:
  CA: $990,053,793
  TX: $388,074,475
  WA: $162,918,291


In [12]:
print("Technology Requirements Summary:")
print(bau_ca.requirements_by_tech(residential=1))

Technology Requirements Summary:
       ev_req_capacity_amp  pv_req_capacity_amp  hp_req_capacity_amp  \
count         57367.000000         57367.000000         57367.000000   
mean             90.284484            10.935362             1.523150   
std              20.544380            12.091422            12.352797   
min              50.000000             0.006703          -235.716667   
25%             100.000000             3.095039            -1.666667   
50%             100.000000             7.160493             0.383333   
75%             100.000000            14.584778             4.000000   
max             150.000000           290.001630           233.566667   

       hpwh_req_capacity_amp  
count           57367.000000  
mean                0.938870  
std                 3.919771  
min               -30.550000  
25%                 0.233333  
50%                 1.566667  
75%                 1.966667  
max                86.216667  


In [15]:
print("\n" + "="*70)
print("ANALYSIS COMPLETE")
print("="*70)
print("\nResults have been saved to: data/parsed_inputs/")
print("  - bicep_results_bau_all_states.csv")
print("  - bicep_cost_summary_bau.csv")


ANALYSIS COMPLETE

Results have been saved to: data/parsed_inputs/
  - bicep_results_bau_all_states.csv
  - bicep_cost_summary_bau.csv


## Summary

This notebook demonstrated BICEP's flexible geographic analysis capabilities:

1. **All States** (`target_states='all'`): Analyze the entire United States
2. **Single State** (`target_states='CA'`): Focus on one specific state
3. **Multiple States** (`target_states=['CA', 'TX', 'WA']`): Analyze a custom set of states

### Key Output Attributes

Each `BicepResults` object provides:
- `total_cost`: Total infrastructure upgrade cost across all years
- `total_residential_costs`: Residential building upgrade costs
- `total_commercial_costs`: Commercial building upgrade costs
- `buildings`: DataFrame with detailed building-level results
- `plot_drivers()`: Visualize cost breakdown by technology driver

### Available Scenarios

- `'bau'`: Business-As-Usual scenario
- `'high'`: High Demand Growth scenario with aggressive adoption

### Database Modes

- `mode='local'`: Uses local SQLite database at `data/bicep.x-stock.db`
- `mode='pnnl'`: Uses Azure SQL Server database (requires network access)